In [ ]:
# ============================================================
# VALIDATION A — BRANCH A
# D8 — Bhutan Land Management Project
# ============================================================
#
# Methodology stage covered:
# Stage 4 — Post-processing and Validation
#
# Compares the preserved Branch A extraction against the
# fixed Stage 1 document-grounded reference dataset.
# ============================================================

import json
import math
import re
import hashlib
import unicodedata

from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment
from difflib import SequenceMatcher


In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D8"
DOCUMENT_NAME = (
    "World Bank — Bhutan - Land Management Project — "
    "Project Information Document (PID), Concept Stage"
)

BRANCH = "A"
BRANCH_NAME = "Direct Ingestion"
INPUT_REPRESENTATION = "Original legacy DOC"

EXPECTED_RECORD_COUNT = 49

EXPECTED_CATEGORY_COUNTS = {
    "Project metadata": 13,
    "Development issue": 10,
    "Bank rationale": 2,
    "Project objective": 3,
    "Project component": 3,
    "Safeguard policy": 6,
    "Financing": 7,
    "Contact information": 5,
}

FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Qualifier",
    "Reporting Period",
    "Source Location",
]

ALIGNMENT_IDENTITY_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Reporting Period",
    "Source Location",
]

PRIMARY_CORRECTNESS_FIELDS = [
    "Value",
    "Unit",
    "Qualifier",
]

DESCRIPTION_DIAGNOSTIC_FIELD = "Description"

MANDATORY_STRING_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Source Location",
]

NULLABLE_STRING_FIELDS = [
    "Unit",
    "Qualifier",
    "Reporting Period",
]

ALLOWED_CATEGORIES = set(EXPECTED_CATEGORY_COUNTS)

REFERENCE_PATH = Path("D8_reference_values.csv")
EXTRACTION_PATH = Path("D8_branch_A_parsed_extraction.json")
TECHNICAL_DIAGNOSTICS_PATH = Path(
    "D8_branch_A_technical_diagnostics.json"
)
EXPERIMENT_METADATA_PATH = Path("D8_branch_A_experiment_metadata.json")

EXPECTED_SOURCE_SHA256 = (
    "61aacfd3138ecfba59fac51d29a970de45d8a909c74b744e756e8a283666c7b5"
)

BLOCK_FIELDS = [
    "Category",
    "Source Location",
]

MATCHING_WEIGHTS = {
    "topic": 0.65,
    "description": 0.25,
    "reporting_period": 0.10,
}

MATCH_SCORE_THRESHOLD = 0.35

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Expected records:", EXPECTED_RECORD_COUNT)
print("Fields:", len(FIELDS))
print("Primary correctness fields:", PRIMARY_CORRECTNESS_FIELDS)


In [ ]:
# ============================================================
# 2. Upload canonical validation inputs
# ============================================================

try:
    from google.colab import files

    required_files = [
        REFERENCE_PATH.name,
        EXTRACTION_PATH.name,
        TECHNICAL_DIAGNOSTICS_PATH.name,
        EXPERIMENT_METADATA_PATH.name,
    ]

    missing_files = [
        filename
        for filename in required_files
        if not Path(filename).exists()
    ]

    if missing_files:
        print("Upload these canonical D8 validation inputs:")
        for filename in missing_files:
            print(" -", filename)

        uploaded = files.upload()

        print("\nUploaded:")
        for filename in uploaded:
            print(" -", filename)
    else:
        print("All required inputs are already available.")

except ImportError:
    print("Not running in Google Colab.")
    print("Place the four required files in the working directory.")

for path in [
    REFERENCE_PATH,
    EXTRACTION_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    EXPERIMENT_METADATA_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required input: {path}")


In [ ]:
# ============================================================
# 3. File hashing utility and input hashes
# ============================================================

def sha256_file(path):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            digest.update(chunk)

    return digest.hexdigest()


REFERENCE_SHA256 = sha256_file(REFERENCE_PATH)
EXTRACTION_SHA256 = sha256_file(EXTRACTION_PATH)
TECHNICAL_DIAGNOSTICS_SHA256 = sha256_file(TECHNICAL_DIAGNOSTICS_PATH)
EXPERIMENT_METADATA_SHA256 = sha256_file(EXPERIMENT_METADATA_PATH)

print("Reference SHA-256:", REFERENCE_SHA256)
print("Extraction SHA-256:", EXTRACTION_SHA256)
print(
    "Technical diagnostics SHA-256:",
    TECHNICAL_DIAGNOSTICS_SHA256
)
print("Experiment-metadata SHA-256:", EXPERIMENT_METADATA_SHA256)


In [ ]:
# ============================================================
# 4. Load fixed Stage 1 reference dataset
# ============================================================

reference_df = pd.read_csv(
    REFERENCE_PATH,
    dtype=object,
    keep_default_na=False,
)

reference_df = reference_df.replace("", None)


def restore_reference_value(value):
    if value is None:
        return None

    text = str(value).strip()

    if text.casefold() in {"one-quarter", "one-third"}:
        return text

    if re.fullmatch(r"-?\d+(?:\.\d+)?", text):
        try:
            number = float(text)
            return int(number) if number.is_integer() else number
        except ValueError:
            pass

    return value


reference_df["Value"] = reference_df["Value"].map(
    restore_reference_value
)

reference_records = reference_df.to_dict(orient="records")

reference_schema_exact = (
    reference_df.columns.tolist() == FIELDS
)

print("Reference records:", len(reference_df))
print("Reference columns:", reference_df.columns.tolist())
print("Reference schema exact:", reference_schema_exact)

display(reference_df.head(12))


In [ ]:
# ============================================================
# 5. Load canonical preserved Branch A extraction
# ============================================================

with EXTRACTION_PATH.open("r", encoding="utf-8") as file:
    parsed_extraction = json.load(file)

top_level_object_valid = isinstance(parsed_extraction, dict)

document_id_correct = (
    top_level_object_valid
    and parsed_extraction.get("document_id") == DOCUMENT_ID
)

branch_correct = (
    top_level_object_valid
    and parsed_extraction.get("branch") == BRANCH
)

records_is_list = (
    top_level_object_valid
    and isinstance(parsed_extraction.get("records"), list)
)

if not records_is_list:
    raise TypeError(
        "Canonical Branch A parsed extraction does not contain a records list."
    )

extracted_records = parsed_extraction["records"]
extracted_df = pd.DataFrame(extracted_records)

print("Top-level object valid:", top_level_object_valid)
print("Document ID correct:", document_id_correct)
print("Branch correct:", branch_correct)
print("Extracted records:", len(extracted_records))

display(extracted_df.head(12))


In [ ]:
# ============================================================
# 6. Load Branch A technical diagnostics and experiment metadata
# ============================================================

with TECHNICAL_DIAGNOSTICS_PATH.open(
    "r",
    encoding="utf-8"
) as file:
    technical_diagnostics = json.load(file)

with EXPERIMENT_METADATA_PATH.open(
    "r",
    encoding="utf-8"
) as file:
    experiment_metadata = json.load(file)

branch_a_structurally_evaluable = bool(
    technical_diagnostics.get(
        "structurally_evaluable",
        False
    )
)

source_hash_matches_stage_1 = (
    metadata_source_hash == EXPECTED_SOURCE_SHA256
)

print(
    "Branch A structurally evaluable:",
    branch_a_structurally_evaluable
)
print(
    "Parsed extraction hash matches Branch A metadata:",
    parsed_extraction_hash_matches_metadata,
)
print(
    "Source hash matches fixed Stage 1 source:",
    source_hash_matches_stage_1,
)

if not parsed_extraction_hash_matches_metadata:
    raise AssertionError(
        "The parsed extraction is not the canonical Branch A extraction "
        "recorded in the experiment metadata."
    )

if not source_hash_matches_stage_1:
    raise AssertionError(
        "The Branch A source hash does not match the fixed Stage 1 source."
    )


In [ ]:
# ============================================================
# 7. Assess extraction record schema
# ============================================================

schema_issue_rows = []
field_order_diagnostic_rows = []

for record_index, record in enumerate(extracted_records):

    if not isinstance(record, dict):
        schema_issue_rows.append({
            "Record Index": record_index,
            "Issue": "Record is not a JSON object",
        })
        continue

    observed_fields = list(record.keys())
    observed_field_set = set(observed_fields)
    expected_field_set = set(FIELDS)

    missing_fields = [
        field for field in FIELDS
        if field not in observed_field_set
    ]

    extra_fields = [
        field for field in observed_fields
        if field not in expected_field_set
    ]

    if missing_fields or extra_fields:
        schema_issue_rows.append({
            "Record Index": record_index,
            "Issue": "Field set mismatch",
            "Missing Fields": ", ".join(missing_fields),
            "Extra Fields": ", ".join(extra_fields),
        })

    if observed_fields != FIELDS:
        field_order_diagnostic_rows.append({
            "Record Index": record_index,
            "Observed Order": observed_fields,
            "Expected Order": FIELDS,
        })


schema_issues_df = pd.DataFrame(schema_issue_rows)
field_order_diagnostics_df = pd.DataFrame(
    field_order_diagnostic_rows
)

record_schema_valid = schema_issues_df.empty

print("Record schema valid:", record_schema_valid)
print("Schema issues:", len(schema_issues_df))
print(
    "Field-order diagnostics:",
    len(field_order_diagnostics_df),
)

if not schema_issues_df.empty:
    display(schema_issues_df)


In [ ]:
# ============================================================
# 8. Reference integrity and extraction content diagnostics
# ============================================================

reference_record_count_valid = (
    len(reference_df) == EXPECTED_RECORD_COUNT
)

reference_category_counts = (
    reference_df["Category"]
    .value_counts()
    .to_dict()
)

reference_category_counts_valid = (
    reference_category_counts == EXPECTED_CATEGORY_COUNTS
)

extracted_record_count = len(extracted_records)

extraction_record_count_valid = (
    extracted_record_count == EXPECTED_RECORD_COUNT
)

extraction_category_counts = dict(
    Counter(
        record.get("Category")
        for record in extracted_records
        if isinstance(record, dict)
    )
)

extraction_category_counts_valid = (
    extraction_category_counts == EXPECTED_CATEGORY_COUNTS
)

print("Reference record count valid:", reference_record_count_valid)
print("Reference category counts valid:", reference_category_counts_valid)
print("Extraction record count valid:", extraction_record_count_valid)
print("Extraction category counts valid:", extraction_category_counts_valid)

if not reference_record_count_valid:
    raise AssertionError(
        "The fixed Stage 1 reference dataset does not contain 49 records."
    )

if not reference_category_counts_valid:
    raise AssertionError(
        "The fixed Stage 1 reference category distribution is invalid."
    )


In [ ]:
# ============================================================
# 9. Null-safe type and mandatory-content diagnostics
# ============================================================

type_issue_rows = []
missing_mandatory_rows = []

for record_index, record in enumerate(extracted_records):

    if not isinstance(record, dict):
        continue

    for field in MANDATORY_STRING_FIELDS:
        value = record.get(field)

        if value is None or value == "":
            missing_mandatory_rows.append({
                "Record Index": record_index,
                "Field": field,
            })
        elif not isinstance(value, str):
            type_issue_rows.append({
                "Record Index": record_index,
                "Field": field,
                "Observed Type": type(value).__name__,
                "Expected Type": "string",
            })

    for field in NULLABLE_STRING_FIELDS:
        value = record.get(field)

        if value is not None and not isinstance(value, str):
            type_issue_rows.append({
                "Record Index": record_index,
                "Field": field,
                "Observed Type": type(value).__name__,
                "Expected Type": "string or null",
            })

    value = record.get("Value")

    if (
        value is not None
        and (
            isinstance(value, bool)
            or not isinstance(value, (str, int, float))
        )
    ):
        type_issue_rows.append({
            "Record Index": record_index,
            "Field": "Value",
            "Observed Type": type(value).__name__,
            "Expected Type": "string, number or null",
        })


type_issues_df = pd.DataFrame(type_issue_rows)
missing_mandatory_fields_df = pd.DataFrame(
    missing_mandatory_rows
)

field_types_valid = type_issues_df.empty
mandatory_fields_complete = missing_mandatory_fields_df.empty

print("Field types valid:", field_types_valid)
print("Mandatory fields complete:", mandatory_fields_complete)


In [ ]:
# ============================================================
# 10. Comparison-only text normalisation and similarity
# ============================================================

def is_missing(value):
    if value is None:
        return True

    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


def normalise_text(value):
    if is_missing(value):
        return None

    text = str(value)

    text = "".join(
        character
        for character in text
        if unicodedata.category(character) != "Cf"
    )

    text = unicodedata.normalize("NFKC", text)

    text = (
        text.replace("’", "'")
        .replace("‘", "'")
        .replace("“", '"')
        .replace("”", '"')
        .replace("–", "-")
        .replace("—", "-")
        .replace("\u00a0", " ")
    )

    text = re.sub(r"\s+", " ", text).strip()
    return text.casefold()


def identity_text(value):
    text = normalise_text(value)

    if text is None:
        return ""

    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def text_similarity(first, second):
    first_text = identity_text(first)
    second_text = identity_text(second)

    if not first_text and not second_text:
        return 1.0

    if not first_text or not second_text:
        return 0.0

    return SequenceMatcher(
        None,
        first_text,
        second_text,
    ).ratio()


def exact_normalised_text_match(first, second):
    return normalise_text(first) == normalise_text(second)


In [ ]:
# ============================================================
# 11. Conservative D8 comparison equivalence rules
# ============================================================

UNIT_EQUIVALENCE_MAP = {
    "%": "percent",
    "percent": "percent",
    "usd million": "usd million",
    "million usd": "usd million",
    "$ million": "usd million",
    "usd millions": "usd million",
}


def canonical_unit(value):
    text = normalise_text(value)

    if text is None:
        return None

    return UNIT_EQUIVALENCE_MAP.get(text, text)


def canonical_period(value):
    return normalise_text(value)


def canonical_qualifier(value):
    return normalise_text(value)


def canonical_topic(value):
    # Intentionally conservative for the first D8 run.
    return normalise_text(value)


def topic_correct(reference_value, extracted_value):
    return (
        canonical_topic(reference_value)
        == canonical_topic(extracted_value)
    )


def unit_correct(reference_value, extracted_value):
    return (
        canonical_unit(reference_value)
        == canonical_unit(extracted_value)
    )


def qualifier_correct(reference_value, extracted_value):
    return (
        canonical_qualifier(reference_value)
        == canonical_qualifier(extracted_value)
    )


def period_correct(reference_value, extracted_value):
    return (
        canonical_period(reference_value)
        == canonical_period(extracted_value)
    )


print(
    "Conservative D8 rules loaded. "
    "No Branch-A-specific Topic aliases are active."
)


In [ ]:
# ============================================================
# 12. Null-safe Value comparison
# ============================================================

def numeric_value(value):
    if is_missing(value) or isinstance(value, bool):
        return None

    if isinstance(value, (int, float)):
        return float(value)

    return None


def value_correct(reference_value, extracted_value):

    if is_missing(reference_value) and is_missing(extracted_value):
        return True

    reference_number = numeric_value(reference_value)
    extracted_number = numeric_value(extracted_value)

    if reference_number is not None and extracted_number is not None:
        return math.isclose(
            reference_number,
            extracted_number,
            rel_tol=0.0,
            abs_tol=1e-12,
        )

    # Do not coerce text such as one-quarter, one-third, codes,
    # dates, phone numbers or textual labels into numbers.
    if isinstance(reference_value, str) and isinstance(extracted_value, str):
        return (
            normalise_text(reference_value)
            == normalise_text(extracted_value)
        )

    return False


In [ ]:
# ============================================================
# 13. Prepare comparison copies and matching blocks
# ============================================================

reference_comparison_df = reference_df.copy()
extracted_comparison_df = pd.DataFrame(
    extracted_records
).copy()

reference_comparison_df["_reference_index"] = np.arange(
    len(reference_comparison_df)
)

extracted_comparison_df["_extraction_index"] = np.arange(
    len(extracted_comparison_df)
)

for frame in [
    reference_comparison_df,
    extracted_comparison_df,
]:
    frame["_block_category"] = frame["Category"].map(
        identity_text
    )

    frame["_block_source_location"] = frame[
        "Source Location"
    ].map(identity_text)


reference_block_counts = (
    reference_comparison_df[
        ["_block_category", "_block_source_location"]
    ]
    .value_counts()
    .to_dict()
)

extracted_block_counts = (
    extracted_comparison_df[
        ["_block_category", "_block_source_location"]
    ]
    .value_counts()
    .to_dict()
)

print("Reference matching blocks:", len(reference_block_counts))
print("Extraction matching blocks:", len(extracted_block_counts))


In [ ]:
# ============================================================
# 14. Identity-only matching score
# ============================================================

def matching_score(reference_row, extracted_row):

    topic_score = text_similarity(
        reference_row["Topic"],
        extracted_row["Topic"],
    )

    description_score = text_similarity(
        reference_row["Description"],
        extracted_row["Description"],
    )

    period_score = text_similarity(
        reference_row["Reporting Period"],
        extracted_row["Reporting Period"],
    )

    total_score = (
        MATCHING_WEIGHTS["topic"] * topic_score
        + MATCHING_WEIGHTS["description"] * description_score
        + MATCHING_WEIGHTS["reporting_period"] * period_score
    )

    return {
        "total": total_score,
        "topic": topic_score,
        "description": description_score,
        "period": period_score,
    }


print(
    "Alignment uses Category + Source Location blocking, then "
    "Topic + Description + Reporting Period identity evidence."
)
print("Value, Unit and Qualifier are excluded from alignment.")


In [ ]:
# ============================================================
# 15. One-to-one Hungarian record alignment
# ============================================================

matched_pairs = []
matched_reference_indices = set()
matched_extraction_indices = set()

reference_blocks = reference_comparison_df.groupby(
    ["_block_category", "_block_source_location"],
    dropna=False,
)

for block_key, reference_block in reference_blocks:

    category_key, source_key = block_key

    extracted_block = extracted_comparison_df.loc[
        (
            extracted_comparison_df["_block_category"]
            == category_key
        )
        & (
            extracted_comparison_df["_block_source_location"]
            == source_key
        )
    ]

    if extracted_block.empty:
        continue

    reference_rows = list(reference_block.iterrows())
    extracted_rows = list(extracted_block.iterrows())

    score_matrix = np.zeros(
        (len(reference_rows), len(extracted_rows)),
        dtype=float,
    )

    component_scores = {}

    for i, (_, reference_row) in enumerate(reference_rows):
        for j, (_, extracted_row) in enumerate(extracted_rows):
            scores = matching_score(
                reference_row,
                extracted_row,
            )

            score_matrix[i, j] = scores["total"]
            component_scores[(i, j)] = scores

    row_indices, column_indices = linear_sum_assignment(
        -score_matrix
    )

    for row_i, column_j in zip(
        row_indices,
        column_indices,
    ):
        score = float(score_matrix[row_i, column_j])

        if score < MATCH_SCORE_THRESHOLD:
            continue

        reference_row = reference_rows[row_i][1]
        extracted_row = extracted_rows[column_j][1]
        scores = component_scores[(row_i, column_j)]

        reference_index = int(
            reference_row["_reference_index"]
        )

        extraction_index = int(
            extracted_row["_extraction_index"]
        )

        matched_pairs.append({
            "reference_index": reference_index,
            "extraction_index": extraction_index,
            "matching_score": score,
            "topic_matching_score": scores["topic"],
            "description_matching_score": scores["description"],
            "period_matching_score": scores["period"],
        })

        matched_reference_indices.add(reference_index)
        matched_extraction_indices.add(extraction_index)


matched_pairs = sorted(
    matched_pairs,
    key=lambda item: item["reference_index"],
)

print("Aligned records:", len(matched_pairs))


In [ ]:
# ============================================================
# 16. Missing and unsupported/unmatched record tables
# ============================================================

all_reference_indices = set(
    reference_comparison_df["_reference_index"].astype(int)
)

all_extraction_indices = set(
    extracted_comparison_df["_extraction_index"].astype(int)
)

missing_reference_indices = sorted(
    all_reference_indices - matched_reference_indices
)

unsupported_extraction_indices = sorted(
    all_extraction_indices - matched_extraction_indices
)

missing_records_df = (
    reference_comparison_df.loc[
        reference_comparison_df["_reference_index"].isin(
            missing_reference_indices
        ),
        FIELDS + ["_reference_index"],
    ]
    .rename(columns={"_reference_index": "Reference Index"})
    .reset_index(drop=True)
)

unsupported_records_df = (
    extracted_comparison_df.loc[
        extracted_comparison_df["_extraction_index"].isin(
            unsupported_extraction_indices
        ),
        FIELDS + ["_extraction_index"],
    ]
    .rename(columns={"_extraction_index": "Extraction Index"})
    .reset_index(drop=True)
)

print("Missing reference records:", len(missing_records_df))
print(
    "Unsupported/unmatched extracted records:",
    len(unsupported_records_df),
)


In [ ]:
# ============================================================
# 17. Field-level comparison of aligned records
# ============================================================

comparison_rows = []

for pair in matched_pairs:

    reference_row = reference_comparison_df.loc[
        reference_comparison_df["_reference_index"]
        == pair["reference_index"]
    ].iloc[0]

    extracted_row = extracted_comparison_df.loc[
        extracted_comparison_df["_extraction_index"]
        == pair["extraction_index"]
    ].iloc[0]

    field_matches = {
        "Category": exact_normalised_text_match(
            reference_row["Category"],
            extracted_row["Category"],
        ),

        "Topic": topic_correct(
            reference_row["Topic"],
            extracted_row["Topic"],
        ),

        "Description": exact_normalised_text_match(
            reference_row["Description"],
            extracted_row["Description"],
        ),

        "Value": value_correct(
            reference_row["Value"],
            extracted_row["Value"],
        ),

        "Unit": unit_correct(
            reference_row["Unit"],
            extracted_row["Unit"],
        ),

        "Qualifier": qualifier_correct(
            reference_row["Qualifier"],
            extracted_row["Qualifier"],
        ),

        "Reporting Period": period_correct(
            reference_row["Reporting Period"],
            extracted_row["Reporting Period"],
        ),

        "Source Location": exact_normalised_text_match(
            reference_row["Source Location"],
            extracted_row["Source Location"],
        ),
    }

    all_primary_fields_match = all(
        field_matches[field]
        for field in PRIMARY_CORRECTNESS_FIELDS
    )

    identity_fields_match = all(
        field_matches[field]
        for field in ALIGNMENT_IDENTITY_FIELDS
    )

    identity_label_difference = (
        all_primary_fields_match
        and not identity_fields_match
    )

    all_mismatched_fields = [
        field
        for field in FIELDS
        if not field_matches[field]
    ]

    primary_mismatched_fields = [
        field
        for field in PRIMARY_CORRECTNESS_FIELDS
        if not field_matches[field]
    ]

    fully_correct = all_primary_fields_match

    output_row = {
        "Reference Index": pair["reference_index"],
        "Extraction Index": pair["extraction_index"],
        "Category": reference_row["Category"],
        "Matching Score": pair["matching_score"],
        "Topic Matching Score": pair["topic_matching_score"],
        "Description Matching Score":
            pair["description_matching_score"],
        "Reporting Period Matching Score":
            pair["period_matching_score"],
        "Description Lexical Similarity":
            text_similarity(
                reference_row["Description"],
                extracted_row["Description"],
            ),
        "all_primary_fields_match":
            bool(all_primary_fields_match),

        "identity_fields_match":
            bool(identity_fields_match),

        "identity_label_difference":
            bool(identity_label_difference),
        "Fully Correct": bool(fully_correct),
        "all_mismatched_fields":
            ", ".join(all_mismatched_fields),
        "primary_mismatched_fields":
            ", ".join(primary_mismatched_fields),
    }

    for field in FIELDS:
        output_row[f"Reference {field}"] = reference_row[field]
        output_row[f"Extracted {field}"] = extracted_row[field]
        output_row[f"{field} Match"] = bool(
            field_matches[field]
        )

    comparison_rows.append(output_row)


comparison_df = pd.DataFrame(comparison_rows)

print("Compared aligned records:", len(comparison_df))

if not comparison_df.empty:
    print(
        "Fully correct primary records:",
        int(comparison_df["Fully Correct"].sum()),
    )

display(comparison_df.head(10))


In [ ]:
# ============================================================
# 18. Split fully correct and discrepant aligned records
# ============================================================

if comparison_df.empty:
    fully_correct_records_df = comparison_df.copy()
    discrepant_records_df = comparison_df.copy()
else:
    fully_correct_records_df = comparison_df.loc[
        comparison_df["Fully Correct"]
    ].copy()

    discrepant_records_df = comparison_df.loc[
        ~comparison_df["Fully Correct"]
    ].copy()

print("Fully correct aligned records:", len(fully_correct_records_df))
print("Discrepant aligned records:", len(discrepant_records_df))

if not discrepant_records_df.empty:
    display(
        discrepant_records_df[
            [
                "Reference Index",
                "Extraction Index",
                "Category",
                "primary_mismatched_fields",
            ]
        ].reset_index(drop=True)
    )


In [ ]:
# ============================================================
# 19. Field-level validation and error summary
# ============================================================

field_rows = []

for field in PRIMARY_CORRECTNESS_FIELDS:

    if comparison_df.empty:
        correct_count = 0
        evaluated_count = 0
        accuracy = None
    else:
        evaluated_count = len(comparison_df)
        correct_count = int(
            comparison_df[f"{field} Match"].sum()
        )

        accuracy = (
            correct_count / evaluated_count
            if evaluated_count > 0
            else None
        )

    role = (
        "diagnostic"
        if field == DESCRIPTION_DIAGNOSTIC_FIELD
        else "primary"
    )

    field_rows.append({
        "Field": field,
        "Aligned Records": evaluated_count,
        "Correct Records": correct_count,
        "Incorrect Records":
            evaluated_count - correct_count,
        "Accuracy": accuracy,
    })


field_validation_df = pd.DataFrame(field_rows)

field_error_summary_df = field_validation_df.loc[
    field_validation_df["Incorrect Records"] > 0
].copy()

display(field_validation_df)


In [ ]:
# ============================================================
# 20. Common record-level validation metrics
# ============================================================

reference_record_count = len(reference_df)
extracted_record_count = len(extracted_records)
aligned_record_count = len(comparison_df)

fully_correct_record_count = (
    int(comparison_df["Fully Correct"].sum())
    if not comparison_df.empty
    else 0
)

discrepant_record_count = (
    aligned_record_count - fully_correct_record_count
)

missing_record_count = len(missing_records_df)
unsupported_record_count = len(unsupported_records_df)

completeness = (
    aligned_record_count / reference_record_count
    if reference_record_count > 0
    else 0.0
)

missing_rate = (
    missing_record_count / reference_record_count
    if reference_record_count > 0
    else 0.0
)

unsupported_rate = (
    unsupported_record_count / extracted_record_count
    if extracted_record_count > 0
    else 0.0
)

record_precision_exact = (
    fully_correct_record_count / extracted_record_count
    if extracted_record_count > 0
    else 0.0
)

record_recall_exact = (
    fully_correct_record_count / reference_record_count
    if reference_record_count > 0
    else 0.0
)

record_f1_exact = (
    2 * record_precision_exact * record_recall_exact
    / (record_precision_exact + record_recall_exact)
    if record_precision_exact + record_recall_exact > 0
    else 0.0
)

discrepancy_rate_among_aligned = (
    discrepant_record_count / aligned_record_count
    if aligned_record_count > 0
    else 0.0
)

primary_correct_count = int(
    field_validation_df[
        "Correct Records"
    ].sum()
)

primary_total_count = int(
    field_validation_df[
        "Aligned Records"
    ].sum()
)

field_accuracy = (
    primary_correct_count / primary_total_count
    if primary_total_count > 0
    else None
)

description_diagnostic_accuracy = (
    float(
        comparison_df[
            "Description Match"
        ].mean()
    )
    if not comparison_df.empty
    else None
)

print("Reference records:", reference_record_count)
print("Extracted records:", extracted_record_count)
print("Aligned records:", aligned_record_count)
print("Fully correct:", fully_correct_record_count)
print("Discrepant:", discrepant_record_count)
print("Missing:", missing_record_count)
print("Unsupported/unmatched:", unsupported_record_count)
print("Completeness:", round(completeness, 4))
print("Exact F1:", round(record_f1_exact, 4))
print(
    "Field accuracy:",
    None
    if field_accuracy is None
    else round(field_accuracy, 4),
)
print(
    "Description diagnostic accuracy:",
    None
    if description_diagnostic_accuracy is None
    else round(description_diagnostic_accuracy, 4),
)

In [ ]:
# ============================================================
# 21. Category-level metrics
# ============================================================

category_rows = []

for category in EXPECTED_CATEGORY_COUNTS:

    expected_records = int(
        (reference_df["Category"] == category).sum()
    )

    extracted_records_category = sum(
        1
        for record in extracted_records
        if record.get("Category") == category
    )

    category_comparison = (
        comparison_df.loc[
            comparison_df["Category"] == category
        ]
        if not comparison_df.empty
        else comparison_df
    )

    aligned_records_category = len(category_comparison)

    fully_correct_category = (
        int(category_comparison["Fully Correct"].sum())
        if not category_comparison.empty
        else 0
    )

    discrepant_category = (
        aligned_records_category - fully_correct_category
    )

    category_completeness = (
        aligned_records_category / expected_records
        if expected_records > 0
        else 0.0
    )

    category_precision = (
        fully_correct_category / extracted_records_category
        if extracted_records_category > 0
        else 0.0
    )

    category_recall = (
        fully_correct_category / expected_records
        if expected_records > 0
        else 0.0
    )

    category_f1 = (
        2 * category_precision * category_recall
        / (category_precision + category_recall)
        if category_precision + category_recall > 0
        else 0.0
    )

    category_rows.append({
        "Category": category,
        "Expected Records": expected_records,
        "Extracted Records": extracted_records_category,
        "Aligned Records": aligned_records_category,
        "Fully Correct Records": fully_correct_category,
        "Discrepant Records": discrepant_category,
        "Completeness": category_completeness,
        "Record Precision Exact": category_precision,
        "Record Recall Exact": category_recall,
        "Record F1 Exact": category_f1,
    })


category_metrics_df = pd.DataFrame(category_rows)
display(category_metrics_df)


In [ ]:
# ============================================================
# 22. Schema validity independently from completeness
# ============================================================

schema_validity = bool(
    branch_a_structurally_evaluable
)

schema_diagnostics = {
    "valid_json": True,
    "top_level_object_valid": bool(top_level_object_valid),
    "document_id_correct": bool(document_id_correct),
    "branch_correct": bool(branch_correct),
    "records_is_list": bool(records_is_list),
    "record_schema_valid": bool(record_schema_valid),
    "field_types_valid": bool(field_types_valid),
    "records_with_structure_issues":
        int(len(schema_issues_df)),
    "records_with_type_issues":
        int(len(type_issues_df)),
    "local_record_schema_valid":
        bool(record_schema_valid),

    "local_field_types_valid":
        bool(field_types_valid),

    "structurally_evaluable":
        bool(branch_a_structurally_evaluable),
    "field_order_diagnostic_count":
        int(len(field_order_diagnostics_df)),
    "schema_validity": bool(schema_validity),
}

content_diagnostics = {
    "reference_record_count_valid":
        bool(reference_record_count_valid),
    "reference_category_counts_valid":
        bool(reference_category_counts_valid),
    "extraction_record_count_valid":
        bool(extraction_record_count_valid),
    "extraction_category_counts_valid":
        bool(extraction_category_counts_valid),
    "mandatory_fields_complete":
        bool(mandatory_fields_complete),
}

print("Schema validity:", schema_validity)
print(json.dumps(schema_diagnostics, indent=2))


In [ ]:
# ============================================================
# 23. D8-specific preservation diagnostics
# ============================================================

def extracted_search_text():
    return json.dumps(
        extracted_records,
        ensure_ascii=False,
    )


search_text = extracted_search_text()

preservation_checks = {
    "one_quarter_preserved":
        "one-quarter" in search_text,
    "one_third_preserved":
        "one-third" in search_text,
    "indigenous_pelple_preserved":
        "Indigenous Pelple" in search_text,
    "borrower_recepient_preserved":
        "BORROWER/RECEPIENT" in search_text,
    "f1_preserved":
        "F1" in search_text,
    "explicit_financing_total_16_5_present":
        any(
            record.get("Category") == "Financing"
            and normalise_text(record.get("Topic")) == "total"
            and value_correct(record.get("Value"), 16.5)
            for record in extracted_records
        ),
    "qualifier_embedded_in_unit_count":
        sum(
            1
            for record in extracted_records
            if isinstance(record.get("Unit"), str)
            and any(
                phrase in normalise_text(record.get("Unit"))
                for phrase in [
                    "at least",
                    "another",
                    "less than",
                    "some",
                    "up to",
                ]
            )
        ),
}

preservation_checks[
    "qualifier_not_embedded_in_unit"
] = (
    preservation_checks[
        "qualifier_embedded_in_unit_count"
    ] == 0
)

print(json.dumps(
    preservation_checks,
    indent=2,
    ensure_ascii=False,
))


In [ ]:
# ============================================================
# 24. Validation summary table
# ============================================================

summary_table = pd.DataFrame([
    {
        "Metric": "Reference records",
        "Value": reference_record_count,
    },
    {
        "Metric": "Extracted records",
        "Value": extracted_record_count,
    },
    {
        "Metric": "Aligned records",
        "Value": aligned_record_count,
    },
    {
        "Metric": "Fully correct records",
        "Value": fully_correct_record_count,
    },
    {
        "Metric": "Discrepant records",
        "Value": discrepant_record_count,
    },
    {
        "Metric": "Missing records",
        "Value": missing_record_count,
    },
    {
        "Metric": "Unsupported/unmatched records",
        "Value": unsupported_record_count,
    },
    {
        "Metric": "Completeness",
        "Value": completeness,
    },
    {
        "Metric": "Exact record F1",
        "Value": record_f1_exact,
    },
    {
        "Metric": "Field accuracy",
        "Value": field_accuracy,
    },
    {
        "Metric": "Description diagnostic accuracy",
        "Value": description_diagnostic_accuracy,
    },
    {
        "Metric": "Schema valid",
        "Value": schema_validity,
    },
])

display(summary_table)


In [ ]:
# ============================================================
# 25. Validation metrics and provenance
# ============================================================

field_accuracy_dictionary = {
    row["Field"]: (
        None
        if pd.isna(row["Accuracy"])
        else float(row["Accuracy"])
    )
    for _, row in field_validation_df.iterrows()
}

category_metrics_dictionary = {
    row["Category"]: {
        "expected_records": int(row["Expected Records"]),
        "extracted_records": int(row["Extracted Records"]),
        "aligned_records": int(row["Aligned Records"]),
        "fully_correct_records":
            int(row["Fully Correct Records"]),
        "discrepant_records":
            int(row["Discrepant Records"]),
        "completeness": float(row["Completeness"]),
        "record_precision_exact":
            float(row["Record Precision Exact"]),
        "record_recall_exact":
            float(row["Record Recall Exact"]),
        "record_f1_exact":
            float(row["Record F1 Exact"]),
    }
    for _, row in category_metrics_df.iterrows()
}

VALIDATION_METRICS = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "input_representation": INPUT_REPRESENTATION,

    "reference_records": int(reference_record_count),
    "extracted_records": int(extracted_record_count),
    "aligned_records": int(aligned_record_count),
    "fully_correct_records":
        int(fully_correct_record_count),
    "discrepant_records":
        int(discrepant_record_count),
    "missing_records":
        int(missing_record_count),
    "unsupported_extracted_records":
        int(unsupported_record_count),

    "completeness": round(float(completeness), 4),
    "missing_rate": round(float(missing_rate), 4),
    "record_precision_exact":
        round(float(record_precision_exact), 4),
    "record_recall_exact":
        round(float(record_recall_exact), 4),
    "record_f1_exact":
        round(float(record_f1_exact), 4),
    "unsupported_rate":
        round(float(unsupported_rate), 4),
    "discrepancy_rate_among_aligned":
        round(float(discrepancy_rate_among_aligned), 4),
    "alignment_identity_fields":
        ALIGNMENT_IDENTITY_FIELDS,

    "primary_correctness_fields":
        PRIMARY_CORRECTNESS_FIELDS,

    "field_accuracy": (
        None
        if field_accuracy is None
        else round(
            float(field_accuracy),
            4,
        )
    ),

    "description_diagnostic_accuracy": (
        None
        if description_diagnostic_accuracy is None
        else round(
            float(description_diagnostic_accuracy),
            4,
        )
    ),

    "field_accuracy_among_aligned":
        field_accuracy_dictionary,

    "schema_validity":
        bool(schema_validity),

    "schema_diagnostics":
        schema_diagnostics,

    "content_diagnostics":
        content_diagnostics,

    "preservation_diagnostics":
        preservation_checks,

    "matching_rules": {
        "blocking_fields":
            BLOCK_FIELDS,
        "one_to_one_assignment":
            "Hungarian linear-sum assignment",
        "matching_score_threshold":
            MATCH_SCORE_THRESHOLD,
        "matching_score_weights":
            MATCHING_WEIGHTS,
        "value_used_for_alignment":
            False,
        "unit_used_for_alignment":
            False,
        "qualifier_used_for_alignment":
            False,
    },

    "comparison_rules": {
        "raw_extraction_modified":
            False,
        "manual_correction_applied":
            False,
        "comparison_normalisation_scope":
            "Comparison copies only",
        "numeric_comparison":
            "Exact represented numeric equality after deterministic parsing",
        "textual_value_comparison":
            "Normalised exact textual equality; textual fractions remain text",
        "topic_correctness":
            "Normalised exact equality in the conservative first run",
        "description":
            "Lexical/string diagnostic only; excluded from primary exact-record correctness because the task permits a concise source-grounded description",
        "unit":
            "Controlled notation equivalence only; qualifier wording is never moved into Unit",
        "qualifier":
            "Dedicated-field normalised exact correctness",
        "reporting_period":
            "Normalised exact correctness in the conservative first run",
        "source_location":
            "Normalised exact correctness",
        "source_typo_policy":
            "Source typos/codes are preserved; repaired spellings are not automatically accepted as equivalent",
        "primary_correctness_fields":
            PRIMARY_CORRECTNESS_FIELDS,
        "d8_equivalence_rules_status":
            (
                "Frozen D8 document/schema-level comparison rules; "
                "no additional Branch-A-derived equivalence rules were required. "
                "Reuse unchanged for Branches A, B and C."
            )
    },

    "category_metrics":
        category_metrics_dictionary,

    "input_provenance": {
        "reference_file":
            REFERENCE_PATH.name,
        "reference_sha256":
            REFERENCE_SHA256,
        "parsed_extraction_file":
            EXTRACTION_PATH.name,
        "parsed_extraction_sha256":
            EXTRACTION_SHA256,
        "technical_diagnostics_file":
            TECHNICAL_DIAGNOSTICS_PATH.name,
        "technical_diagnostics_sha256":
            TECHNICAL_DIAGNOSTICS_SHA256,
        "experiment_metadata_file":
            EXPERIMENT_METADATA_PATH.name,
        "experiment_metadata_sha256":
            EXPERIMENT_METADATA_SHA256,
        "branch_a_structurally_evaluable":
            bool(branch_a_structurally_evaluable),
        "parsed_extraction_hash_matches_metadata":
            bool(parsed_extraction_hash_matches_metadata),
        "source_hash_matches_stage_1":
            bool(source_hash_matches_stage_1),
    },

}

print(json.dumps(
    VALIDATION_METRICS,
    ensure_ascii=False,
    indent=2,
))

In [ ]:
# ============================================================
# 26. Validation metadata and conclusion
# ============================================================

VALIDATION_METADATA = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "validation_type":
        "Post-extraction reference-value agreement",
    "reference_dataset":
        REFERENCE_PATH.name,
    "extraction_dataset":
        EXTRACTION_PATH.name,
    "framework":
        (
            "Fixed Stage 1 reference -> canonical Branch A extraction -> "
            "provenance/schema checks -> comparison-only normalisation -> "
            "outcome-independent one-to-one alignment -> field comparison -> "
            "record classification -> common metrics -> D8 diagnostics"
        ),
    "description_primary_correctness":
        False,
    "description_policy":
        (
            "Description is evaluated diagnostically because the fixed "
            "extraction task asks for a concise source-grounded description "
            "rather than one uniquely prescribed string. The same rule must "
            "be used for D8 Branches A, B and C."
        ),
    "notes": (
        "Alignment uses descriptive identity fields only. "
        "Formal substantive correctness is evaluated over "
        "Value, Unit and Qualifier. Description remains a "
        "diagnostic field because the extraction task permits "
        "a concise source-grounded description."
    )
}


VALIDATION_CONCLUSION = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "schema_valid":
        bool(schema_validity),
    "content_evaluable":
        True,
    "validation_completed":
        True,
    "equivalence_rules_frozen":
        True,

    "next_step":
        (
            "Reuse the frozen D8 validation rules unchanged for "
            "Branches B and C."
        ),
}


In [ ]:
# ============================================================
# 27. Define output paths and export Validation A outputs
# ============================================================

OUTPUT_PATHS = {
    "summary_json":
        Path("D8_branch_A_validation_summary.json"),
    "summary_csv":
        Path("D8_branch_A_validation_summary.csv"),
    "detailed_csv":
        Path("D8_branch_A_validation_detailed.csv"),
    "fully_correct_csv":
        Path("D8_branch_A_fully_correct_records.csv"),
    "discrepant_csv":
        Path("D8_branch_A_discrepant_records.csv"),
    "missing_csv":
        Path("D8_branch_A_missing_records.csv"),
    "unsupported_csv":
        Path("D8_branch_A_unsupported_records.csv"),
    "schema_issues_csv":
        Path("D8_branch_A_schema_issues.csv"),
    "field_order_csv":
        Path("D8_branch_A_field_order_diagnostics.csv"),
    "type_issues_csv":
        Path("D8_branch_A_type_issues.csv"),
    "missing_mandatory_csv":
        Path("D8_branch_A_missing_mandatory_fields.csv"),
    "field_validation_csv":
        Path("D8_branch_A_field_validation.csv"),
    "field_error_summary_csv":
        Path("D8_branch_A_field_error_summary.csv"),
    "category_metrics_csv":
        Path("D8_branch_A_category_metrics.csv"),
    "metadata_json":
        Path("D8_branch_A_validation_metadata.json"),
    "conclusion_json":
        Path("D8_branch_A_validation_conclusion.json"),
}

with OUTPUT_PATHS["summary_json"].open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        VALIDATION_METRICS,
        file,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )

summary_table.to_csv(
    OUTPUT_PATHS["summary_csv"],
    index=False,
    encoding="utf-8-sig",
)

comparison_df.to_csv(
    OUTPUT_PATHS["detailed_csv"],
    index=False,
    encoding="utf-8-sig",
)

fully_correct_records_df.to_csv(
    OUTPUT_PATHS["fully_correct_csv"],
    index=False,
    encoding="utf-8-sig",
)

discrepant_records_df.to_csv(
    OUTPUT_PATHS["discrepant_csv"],
    index=False,
    encoding="utf-8-sig",
)

missing_records_df.to_csv(
    OUTPUT_PATHS["missing_csv"],
    index=False,
    encoding="utf-8-sig",
)

unsupported_records_df.to_csv(
    OUTPUT_PATHS["unsupported_csv"],
    index=False,
    encoding="utf-8-sig",
)

schema_issues_df.to_csv(
    OUTPUT_PATHS["schema_issues_csv"],
    index=False,
    encoding="utf-8-sig",
)

field_order_diagnostics_df.to_csv(
    OUTPUT_PATHS["field_order_csv"],
    index=False,
    encoding="utf-8-sig",
)

type_issues_df.to_csv(
    OUTPUT_PATHS["type_issues_csv"],
    index=False,
    encoding="utf-8-sig",
)

missing_mandatory_fields_df.to_csv(
    OUTPUT_PATHS["missing_mandatory_csv"],
    index=False,
    encoding="utf-8-sig",
)

field_validation_df.to_csv(
    OUTPUT_PATHS["field_validation_csv"],
    index=False,
    encoding="utf-8-sig",
)

field_error_summary_df.to_csv(
    OUTPUT_PATHS["field_error_summary_csv"],
    index=False,
    encoding="utf-8-sig",
)

category_metrics_df.to_csv(
    OUTPUT_PATHS["category_metrics_csv"],
    index=False,
    encoding="utf-8-sig",
)

with OUTPUT_PATHS["metadata_json"].open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        VALIDATION_METADATA,
        file,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )

with OUTPUT_PATHS["conclusion_json"].open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        VALIDATION_CONCLUSION,
        file,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )

print("D8 Validation A outputs exported.")


In [ ]:
# ============================================================
# 28. Final validation consistency checks
# ============================================================

assert reference_record_count_valid
assert reference_category_counts_valid
assert parsed_extraction_hash_matches_metadata
assert source_hash_matches_stage_1
assert all(path.exists() for path in OUTPUT_PATHS.values())

# Schema validity does not depend on record count or category completeness.
assert schema_validity == bool(
    branch_a_structurally_evaluable
)

print("D8 Validation A revised notebook completed successfully.")
print("Schema valid:", schema_validity)
print("Aligned:", aligned_record_count)
print("Missing:", missing_record_count)
print("Unsupported/unmatched:", unsupported_record_count)
print("Fully correct:", fully_correct_record_count)
print("Discrepant:", discrepant_record_count)
print("Exact F1:", round(record_f1_exact, 4))
print(
    "Field accuracy:",
    None
    if field_accuracy is None
    else round(field_accuracy, 4),
)
print(
    "Description diagnostic accuracy:",
    None
    if description_diagnostic_accuracy is None
    else round(description_diagnostic_accuracy, 4),
)


In [ ]:
# ============================================================
# 29. List generated outputs
# ============================================================

print("Generated files:")

for label, path in OUTPUT_PATHS.items():
    print(
        f" - {label}: {path.name} "
        f"| exists: {path.exists()}"
    )
